# Harfi: Qwen3-0.6B v2 on spelling v2.0 and round 6

Qwen v2 ([ADR 0040](../docs/decisions/0040-spelling-v2.md)): **Qwen3-0.6B** with the size study's recipe (LoRA rank 32,
alpha 64, lr 2e-4, best epoch by validation loss kept) with 2 epochs instead of 3: v1's best was epoch 2. Only the data changes: the `harfi-v6` export (144,986
examples), where every row follows spelling v2.0 (ا for أ/إ, plural و, ليه/ليش, هي) and round 6 adds 11,393 AI-written rows
aimed at Qwen v1's measured errors (docs/SIZE_STUDY_ERRORS.md).

Qwen3 trains in fp16 on the T4. Its thinking mode is turned off in the prompt, as for v1.

Licence: Apache-2.0.

**Output:** `arabizi-qwen3-0.6b-v2-Q4_K_M.gguf` and `arabizi-qwen3-0.6b-v2-Q8_0.gguf`, their SHA-256 files and `run.json`
(with the exact prompt the engine needs).

**How to run:**
1. Upload `build/arabizi-train-v6.zip` as a new Kaggle dataset `harfi-v6`, and add it (only it): Add Input -> your dataset.
2. Settings -> Accelerator -> **GPU T4** · **Internet on**.
3. If the Qwen v2 changes are not merged into `main` yet, set `REF` below to their branch.
4. **Save Version -> Save & Run All (Commit)**: about 4.7 hours in the background (under Kaggle's 12-hour limit).
5. From the version's **Output** tab, **Download all** and send the zip. It is scored on the laptop against v4 and Qwen v1
   (their outputs converted to spelling v2.0 first, so the comparison is fair).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU: the same effective batch (32) as v4


In [ ]:
REF = "main"  # or the Qwen v2 branch until it is merged
# Fresh copy of the code on every run (an old clone would otherwise be kept and run).
!rm -rf /kaggle/working/repo
!git clone --quiet --depth 1 --branch "$REF" https://github.com/Mustafa-Abbara/arabizi-keyboard /kaggle/working/repo
!git -C /kaggle/working/repo log -1 --format='repo commit: %h %s'
!pip install --quiet -r /kaggle/working/repo/train/requirements.txt


In [ ]:
import glob
import json
import subprocess
import sys

exports = glob.glob("/kaggle/input/**/export.json", recursive=True)
assert len(exports) == 1, f"add exactly one dataset with export.json (found {exports})"
DATA = os.path.dirname(exports[0])
with open(exports[0]) as f:
    summary = json.load(f)
print(DATA, "|", summary["rows_total"], "examples | AI share", summary["ai_share"], "| prompt v", summary["prompt_version"])
assert summary["rows_total"] == 144986, "this should be the harfi-v6 export (spelling v2.0 + round 6)"
OUT = "/kaggle/working/out"
REPO = "/kaggle/working/repo"


In [ ]:
# v1's settings (rank 32, alpha 64, lr 2e-4), 2 epochs: v1's best was epoch 2 (checkpoint-7274), and 3 epochs on this
# larger export would take about 7 hours, too close to the quota left.
BASE, TAG, NAME = "qwen3-0.6b", "v2", "arabizi-qwen3-0.6b-v2"

print(f"=== {BASE} {TAG}", flush=True)
# check=True: a failed run stops the notebook instead of exporting nothing.
subprocess.run([sys.executable, f"{REPO}/train/finetune_lora.py", "--base", BASE, "--data", DATA, "--out", OUT,
                "--tag", TAG, "--rank", "32", "--alpha", "64", "--epochs", "2"], check=True)
run = max(glob.glob(f"{OUT}/*-{BASE}-lora-{TAG}"))
subprocess.run(["bash", f"{REPO}/train/export_gguf.sh", f"{run}/merged", NAME, run, "Q4_K_M Q8_0"], check=True)
subprocess.run(["rm", "-rf", f"{run}/merged", f"{run}/checkpoints"], check=True)


## Results and download

The best validation loss is only a first look (v1's is in its run.json; the data differs, so compare accuracy, not loss); accuracy on real messages, measured on the laptop,
decides. From the version's **Output** tab, use **Download all** (one zip).


In [ ]:
for path in sorted(glob.glob(f"{OUT}/*-lora-*/run.json")):
    with open(path) as f:
        r = json.load(f)
    print(f"{r['run_id']:45} best validation loss {r['best_eval_loss']:.4f}  ({r['minutes']} min, {r['precision']})")
    for s in r["public_samples"]:
        print(f"  {s['arabizi']}\n    -> {s['output']}\n    want {s['reference']}")
!find "$OUT" -maxdepth 2 -type f -printf "%10s  %p\n"
